<a href="https://colab.research.google.com/github/korkutanapa/DCASE2025TASK2/blob/main/ORJ_DCASE_FEATURE_SELECTON_FOR_UNSEEN_DATA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
DCASE 2025 Task 2
FROZEN-68 -> 1200-POINT TRANSDUCTIVE FEATURE-SUBSET SELECTION ONLY
DUAL RAW kNN: SOURCE k=10, TARGET k=3, FINAL=min(dS,dT)
====================================================================

PURPOSE
-------
For each unseen/evaluation machine:

    TRAIN = 990 source-normal + 10 target-normal = 1000 KNOWN NORMALS
    TEST  = 200 unlabeled recordings

The code transfers the fixed development-derived pool of 68 TDA descriptors,
fits pooled-normal train MinMax scaling (without clipping test values), keeps
source and target normal reference banks separate, and evaluates candidate
feature subsets using

    dS(x;S) = mean Euclidean distance to 10 source-normal neighbors
    dT(x;S) = mean Euclidean distance to  3 target-normal neighbors
    A(x;S)  = min(dS(x;S), dT(x;S)).

Known-normal train samples are scored with leave-one-out in their own domain.
All 1000 known-normal train recordings and 200 unlabeled test recordings are
used in the 1200-point label-free/transductive subset-selection objective.
No test anomaly labels and no test domain labels are used.

SEARCH
------
    D=1      : exhaustive 68 singles
    D=2      : exhaustive C(68,2)=2278 pairs
    D=3..20  : beam search + random injections

Candidate subsets are compared within dimension and then across dimensions
using the same dimension-normalized label-free criterion as the original code.


OUTPUT
------
The output directory contains feature-selection reports only, including:
    selected_top10_by_machine.csv/json
    selected_top1_by_machine.csv/json
    selected_top1_features_long.csv
    best_by_dimension.csv
    feature_effectiveness_ranking.csv
    global_feature_effectiveness_ranking.csv
    search_method_config.json
    per_machine/<machine>/<machine>_top10_subsets.csv
    per_machine/<machine>/<machine>_best_by_dimension.csv
    per_machine/<machine>/<machine>_feature_effectiveness_ranking.csv

A ZIP archive of the report directory is also created for convenience.
"""

from __future__ import annotations

import glob
import json
import math
import os
import random
import shutil
import time
import warnings
from dataclasses import dataclass
from itertools import combinations
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore", category=RuntimeWarning)


# =============================================================================
# 0. CONFIG -- CHANGE THESE IF YOU WANT A DEEPER / FASTER SEARCH
# =============================================================================

MACHINE_TYPES = [
    "AutoTrash",
    "BandSealer",
    "CoffeeGrinder",
    "HomeCamera",
    "Polisher",
    "ScrewFeeder",
    "ToyPet",
    "ToyRCCar",
]

BEST_FEATURE_POOL_68 = [
    "H0_landscape_auc",
    "H0_landscape_l1",
    "H0_landscape_l2",
    "H0_landscape_layer5_auc",
    "H0_lifetime_skew",
    "H0_pimage_entropy",
    "H0_pimage_min",
    "H1_betti_max",
    "H1_birth_iqr",
    "H1_birth_var",
    "H1_death_iqr",
    "H1_death_max",
    "H1_death_min",
    "H1_death_var",
    "H1_min_lifetime",
    "H1_min_midlife",
    "H1_q25_lifetime",
    "H1_to_H0_num_points_ratio",
    "H1_weighted_midlife_std",
    "H0_betti_max",
    "H0_betti_num_peaks",
    "H0_landscape_layer2_max",
    "H0_landscape_layer3_max",
    "H0_mean_birth_death_ratio",
    "H0_range_lifetime",
    "H0_top3_share",
    "H1_landscape_layer2_auc",
    "H1_landscape_layer2_mean",
    "H1_landscape_layer3_mean",
    "H1_landscape_layer4_auc",
    "H1_max_lifetime",
    "H1_minus_H0_entropy",
    "H1_silhouette_std",
    "H1_std_birth_death_ratio",
    "H0_birth_skew",
    "H0_death_skew",
    "H0_persistence_entropy",
    "H0_top1_share",
    "H1_normalized_persistence_entropy",
    "H1_num_points",
    "H1_to_H0_entropy_ratio",
    "H0_death_min",
    "H0_landscape_entropy",
    "H1_death_std",
    "H1_landscape_layer2_max",
    "H1_pimage_min",
    "H0_betti_l1",
    "H0_birth_max",
    "H0_death_q25",
    "H0_landscape_layer5_max",
    "H0_max_midlife",
    "H0_median_lifetime",
    "H0_num_points",
    "H0_pimage_energy",
    "H1_birth_max",
    "H1_silhouette_entropy",
    "H1_tail_share_q90",
    "H1_tail_share_q95",
    "H0_landscape_layer1_mean",
    "H1_landscape_layer1_auc",
    "H1_to_H0_max_lifetime_ratio",
    "H0_birth_kurtosis",
    "H0_landscape_layer4_max",
    "H0_q75_lifetime",
    "H1_betti_l2",
    "H1_betti_std",
    "H1_mean_birth_death_ratio",
    "H1_persistence_entropy",
]

# Reference neighborhoods requested by the user.
SOURCE_K = 10
TARGET_K = 3

# Feature-subset dimensions.
MAX_DIM = 20

# Number of highest-ranked label-free subsets to report per machine.
TOP_N_SUBSETS = 10

# 1200-point high/far group is allowed to float in this range.
# Expected anomaly count is ~100, but 50..150 is intentionally flexible.
HIGH_GROUP_MIN = 50
HIGH_GROUP_MAX = 150
EXPECTED_HIGH_GROUP = 100

# Scaling: fit on pooled 990+10 normal train only, no clipping on test.
SCALER_MODE = "minmax_train"  # currently implemented/frozen final mode

# Divide Euclidean distances by sqrt(D) for numerical comparability across D.
# This does NOT alter ranking within a fixed subset/dimension.
DISTANCE_DIM_NORMALIZE = True

# Search depth. Increase these for an even deeper run.
BEAM_WIDTH = 24
EXPANSION_POOL_SIZE = 44
TOP_SINGLES_FOR_POOL = 30
TOP_PAIRS_FOR_POOL = 200
RANDOM_INJECTIONS_PER_DIM = 80
CALIBRATION_RANDOM_PER_DIM = 60
TOP_KEEP_PER_DIM = 80

# Random seed for reproducibility.
RANDOM_SEED = 20260819

# 1200-point objective weights.
# Geometry terms are log-compressed before weighting.
W_SEPARATION = 1.00
W_BOUNDARY_GAP = 0.65
W_TAIL_SEPARATION = 0.55
W_HIGH_TEST_PURITY = 1.50
W_TEST_CAPTURE = 0.45
W_KNOWN_NORMAL_CONTAM = 2.00
W_HIGH_TRAIN_FRACTION = 6.00
W_SIZE_PREFERENCE = 0.25

# Numerical safety.
EPS = 1e-12
MAX_Z = 50.0


# Output / Colab behavior.
AUTO_DOWNLOAD_ZIP = True

# Auto-detect default input root.
if os.path.isdir("/content"):
    INPUT_DIR = "/content"
    OUTPUT_DIR = "/content/dcase2025_FINAL_68_feature_selection_reports"
else:
    INPUT_DIR = "/mnt/data"
    OUTPUT_DIR = "/mnt/data/dcase2025_FINAL_68_feature_selection_reports"

PER_MACHINE_DIR = os.path.join(OUTPUT_DIR, "per_machine")


# =============================================================================
# 1. BASIC UTILITIES
# =============================================================================

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def clean_output_dir() -> None:
    if os.path.isdir(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
    ensure_dir(OUTPUT_DIR)
    ensure_dir(PER_MACHINE_DIR)


def normalize_file_id(value: object) -> str:
    s = str(value).strip().replace("\\", "/")
    return os.path.basename(s)


def infer_train_domain(series: pd.Series) -> np.ndarray:
    s = series.astype(str).str.lower()
    out = np.full(len(s), "", dtype=object)
    out[s.str.contains(r"(?:^|_)source(?:_|$)", regex=True)] = "source"
    out[s.str.contains(r"(?:^|_)target(?:_|$)", regex=True)] = "target"
    return out


def robust_scale_1d(values: np.ndarray) -> float:
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return 1.0

    q25, q75 = np.quantile(x, [0.25, 0.75])
    s = float(q75 - q25)
    if s > EPS:
        return s

    q10, q90 = np.quantile(x, [0.10, 0.90])
    s = float(q90 - q10)
    if s > EPS:
        return s

    s = float(np.std(x))
    if s > EPS:
        return s

    med_abs = float(np.median(np.abs(x))) if len(x) else 1.0
    return max(0.05 * max(med_abs, 1.0), EPS)


def robust_z(value: float, baseline_values: Sequence[float]) -> float:
    x = np.asarray(list(baseline_values), dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return 0.0
    med = float(np.median(x))
    scale = robust_scale_1d(x)
    z = (float(value) - med) / max(scale, EPS)
    return float(np.clip(z, -MAX_Z, MAX_Z))


def safe_log_positive(x: float) -> float:
    return float(np.log1p(max(0.0, float(x))))


def set_global_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


def unique_tuples(items: Iterable[Tuple[int, ...]]) -> List[Tuple[int, ...]]:
    seen = set()
    out = []
    for item in items:
        t = tuple(sorted(int(v) for v in item))
        if t not in seen:
            seen.add(t)
            out.append(t)
    return out


# =============================================================================
# 2. FILE DISCOVERY
# =============================================================================

def find_machine_files(machine: str) -> Tuple[str, str]:
    """Find newest train (_thr) and test (non-_thr) XLSX for one machine."""
    all_xlsx = glob.glob(os.path.join(INPUT_DIR, "*.xlsx"))
    mk = machine.lower()

    machine_files = [
        p for p in all_xlsx
        if mk in os.path.basename(p).lower()
        and not os.path.basename(p).startswith("~$")
    ]

    train_matches = [
        p for p in machine_files
        if "_thr" in os.path.basename(p).lower()
    ]

    test_matches = [
        p for p in machine_files
        if "_thr" not in os.path.basename(p).lower()
    ]

    train_matches = sorted(
        train_matches,
        key=lambda p: (os.path.getmtime(p), p),
        reverse=True,
    )
    test_matches = sorted(
        test_matches,
        key=lambda p: (os.path.getmtime(p), p),
        reverse=True,
    )

    if not train_matches:
        raise FileNotFoundError(
            f"{machine}: no *_thr*.xlsx train file found under {INPUT_DIR}"
        )
    if not test_matches:
        raise FileNotFoundError(
            f"{machine}: no non-_thr XLSX test file found under {INPUT_DIR}"
        )

    return train_matches[0], test_matches[0]


# =============================================================================
# 3. DATA PREPARATION
# =============================================================================

@dataclass
class PreparedMachine:
    machine: str
    feature_names: List[str]
    train_path: str
    test_path: str

    train_file_ids: np.ndarray
    test_file_ids: np.ndarray
    train_domains: np.ndarray

    X_train_raw: np.ndarray
    X_test_raw: np.ndarray
    X_train_scaled: np.ndarray
    X_test_scaled: np.ndarray
    X_source: np.ndarray
    X_target: np.ndarray

    source_train_indices: np.ndarray
    target_train_indices: np.ndarray

    impute_median: np.ndarray
    scale_min: np.ndarray
    scale_max: np.ndarray
    scale_range: np.ndarray


def numeric_feature_frame(df: pd.DataFrame, features: Sequence[str]) -> np.ndarray:
    arr = (
        df[list(features)]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=np.float64)
    )
    arr[~np.isfinite(arr)] = np.nan
    return arr


def prepare_machine(machine: str) -> PreparedMachine:
    train_path, test_path = find_machine_files(machine)

    print("\n" + "=" * 120)
    print(f"PREPARING {machine}")
    print("=" * 120)
    print("train:", os.path.basename(train_path))
    print("test :", os.path.basename(test_path))

    train_df = pd.read_excel(train_path)
    test_df = pd.read_excel(test_path)

    missing_train = [f for f in BEST_FEATURE_POOL_68 if f not in train_df.columns]
    missing_test = [f for f in BEST_FEATURE_POOL_68 if f not in test_df.columns]
    if missing_train or missing_test:
        raise ValueError(
            f"{machine}: frozen-68 mismatch. "
            f"missing_train={missing_train}, missing_test={missing_test}"
        )

    # File IDs are needed for official prediction output only.
    if "file_id" in train_df.columns:
        train_ids = train_df["file_id"].map(normalize_file_id).to_numpy(dtype=str)
    elif "file_path" in train_df.columns:
        train_ids = train_df["file_path"].map(normalize_file_id).to_numpy(dtype=str)
    else:
        raise ValueError(f"{machine}: train needs file_id or file_path")

    if "file_id" in test_df.columns:
        test_ids = test_df["file_id"].map(normalize_file_id).to_numpy(dtype=str)
    elif "file_path" in test_df.columns:
        test_ids = test_df["file_path"].map(normalize_file_id).to_numpy(dtype=str)
    else:
        raise ValueError(f"{machine}: test needs file_id or file_path")

    # Infer source/target ONLY for train. Test domain is deliberately not used.
    domain_candidates = []
    if "file_id" in train_df.columns:
        domain_candidates.append(train_df["file_id"])
    if "file_path" in train_df.columns:
        domain_candidates.append(train_df["file_path"])

    train_domains = np.full(len(train_df), "", dtype=object)
    for series in domain_candidates:
        inferred = infer_train_domain(series)
        fill = train_domains == ""
        train_domains[fill] = inferred[fill]

    if np.any(train_domains == ""):
        bad = np.where(train_domains == "")[0][:10].tolist()
        raise ValueError(f"{machine}: cannot infer train domain at rows {bad}")

    src_mask = train_domains == "source"
    tgt_mask = train_domains == "target"

    n_src = int(src_mask.sum())
    n_tgt = int(tgt_mask.sum())
    n_test = len(test_df)

    print(f"rows: source-normal={n_src}, target-normal={n_tgt}, unlabeled-test={n_test}")

    if n_src != 990 or n_tgt != 10:
        print(
            f"WARNING: expected 990 source + 10 target normals; got {n_src}+{n_tgt}. "
            "Code will continue using the detected known-normal banks."
        )
    if n_test != 200:
        print(f"WARNING: expected 200 test rows; got {n_test}.")

    # We never read/use test labels here. If a label column exists, it is ignored.
    # Train label is also unnecessary because these files are assumed normal train.
    X_train_raw = numeric_feature_frame(train_df, BEST_FEATURE_POOL_68)
    X_test_raw = numeric_feature_frame(test_df, BEST_FEATURE_POOL_68)

    # POOLED 1000-NORMAL imputation and min-max fitting.
    impute_median = np.nanmedian(X_train_raw, axis=0)
    bad_med = ~np.isfinite(impute_median)
    if np.any(bad_med):
        bad_names = [BEST_FEATURE_POOL_68[i] for i in np.where(bad_med)[0]]
        raise ValueError(f"{machine}: no finite train values for features {bad_names}")

    def impute(arr: np.ndarray) -> np.ndarray:
        out = np.asarray(arr, dtype=float).copy()
        bad = ~np.isfinite(out)
        if np.any(bad):
            rr, cc = np.where(bad)
            out[rr, cc] = impute_median[cc]
        return out

    X_train_imp = impute(X_train_raw)
    X_test_imp = impute(X_test_raw)

    if SCALER_MODE != "minmax_train":
        raise ValueError(f"Unsupported SCALER_MODE={SCALER_MODE}")

    scale_min = np.min(X_train_imp, axis=0)
    scale_max = np.max(X_train_imp, axis=0)
    scale_range = scale_max - scale_min

    bad_range = (~np.isfinite(scale_range)) | (scale_range <= EPS)
    if np.any(bad_range):
        bad_names = [BEST_FEATURE_POOL_68[i] for i in np.where(bad_range)[0]]
        raise ValueError(
            f"{machine}: constant/invalid frozen features under pooled train MinMax: {bad_names}"
        )

    X_train_scaled = (X_train_imp - scale_min) / scale_range
    X_test_scaled = (X_test_imp - scale_min) / scale_range
    # IMPORTANT: no clipping. Test can be <0 or >1.

    src_idx = np.where(src_mask)[0]
    tgt_idx = np.where(tgt_mask)[0]
    X_source = X_train_scaled[src_idx]
    X_target = X_train_scaled[tgt_idx]

    prepared = PreparedMachine(
        machine=machine,
        feature_names=list(BEST_FEATURE_POOL_68),
        train_path=train_path,
        test_path=test_path,
        train_file_ids=train_ids,
        test_file_ids=test_ids,
        train_domains=train_domains,
        X_train_raw=X_train_raw,
        X_test_raw=X_test_raw,
        X_train_scaled=X_train_scaled,
        X_test_scaled=X_test_scaled,
        X_source=X_source,
        X_target=X_target,
        source_train_indices=src_idx,
        target_train_indices=tgt_idx,
        impute_median=impute_median,
        scale_min=scale_min,
        scale_max=scale_max,
        scale_range=scale_range,
    )

    return prepared


# =============================================================================
# 4. DUAL RAW kNN SCORING
# =============================================================================

def _query_mean(tree: cKDTree, X: np.ndarray, k: int) -> np.ndarray:
    k_eff = max(1, min(int(k), int(tree.n)))
    d, _ = tree.query(X, k=k_eff, workers=-1)
    d = np.asarray(d, dtype=float)
    if d.ndim == 1:
        d = d[:, None]
    return np.mean(d, axis=1)


def _loo_mean(tree: cKDTree, X: np.ndarray, k: int) -> np.ndarray:
    n = len(X)
    if n < 2:
        raise ValueError("LOO kNN needs at least 2 samples")
    k_eff = min(int(k), n - 1)
    d, _ = tree.query(X, k=k_eff + 1, workers=-1)
    d = np.asarray(d, dtype=float)
    if d.ndim == 1:
        d = d[:, None]
    # First neighbor is self at distance 0.
    return np.mean(d[:, 1:k_eff + 1], axis=1)


@dataclass
class ScoreBundle:
    subset: Tuple[int, ...]
    dim: int

    source_train_dS: np.ndarray
    source_train_dT: np.ndarray
    source_train_score: np.ndarray

    target_train_dS: np.ndarray
    target_train_dT: np.ndarray
    target_train_score: np.ndarray

    test_dS: np.ndarray
    test_dT: np.ndarray
    test_score: np.ndarray

    all_scores: np.ndarray
    all_is_train: np.ndarray
    all_is_test: np.ndarray
    all_origin: np.ndarray
    all_file_ids: np.ndarray


def compute_score_bundle(data: PreparedMachine, subset: Tuple[int, ...]) -> ScoreBundle:
    subset = tuple(sorted(int(j) for j in subset))
    idx = np.asarray(subset, dtype=int)
    dim = len(subset)
    if dim < 1:
        raise ValueError("empty subset")

    Xs = np.ascontiguousarray(data.X_source[:, idx], dtype=np.float64)
    Xt = np.ascontiguousarray(data.X_target[:, idx], dtype=np.float64)
    Xq = np.ascontiguousarray(data.X_test_scaled[:, idx], dtype=np.float64)

    src_tree = cKDTree(Xs)
    tgt_tree = cKDTree(Xt)

    # Source train normal scores.
    src_dS = _loo_mean(src_tree, Xs, SOURCE_K)
    src_dT = _query_mean(tgt_tree, Xs, TARGET_K)

    # Target train normal scores.
    tgt_dS = _query_mean(src_tree, Xt, SOURCE_K)
    tgt_dT = _loo_mean(tgt_tree, Xt, TARGET_K)

    # Test scores.
    test_dS = _query_mean(src_tree, Xq, SOURCE_K)
    test_dT = _query_mean(tgt_tree, Xq, TARGET_K)

    if DISTANCE_DIM_NORMALIZE:
        denom = math.sqrt(float(dim))
        src_dS = src_dS / denom
        src_dT = src_dT / denom
        tgt_dS = tgt_dS / denom
        tgt_dT = tgt_dT / denom
        test_dS = test_dS / denom
        test_dT = test_dT / denom

    src_score = np.minimum(src_dS, src_dT)
    tgt_score = np.minimum(tgt_dS, tgt_dT)
    test_score = np.minimum(test_dS, test_dT)

    all_scores = np.concatenate([src_score, tgt_score, test_score])
    n_src = len(src_score)
    n_tgt = len(tgt_score)
    n_test = len(test_score)

    all_is_train = np.concatenate([
        np.ones(n_src + n_tgt, dtype=bool),
        np.zeros(n_test, dtype=bool),
    ])
    all_is_test = ~all_is_train
    all_origin = np.asarray(
        ["source_train"] * n_src
        + ["target_train"] * n_tgt
        + ["test"] * n_test,
        dtype=object,
    )

    src_ids = data.train_file_ids[data.source_train_indices]
    tgt_ids = data.train_file_ids[data.target_train_indices]
    all_ids = np.concatenate([src_ids, tgt_ids, data.test_file_ids])

    return ScoreBundle(
        subset=subset,
        dim=dim,
        source_train_dS=src_dS,
        source_train_dT=src_dT,
        source_train_score=src_score,
        target_train_dS=tgt_dS,
        target_train_dT=tgt_dT,
        target_train_score=tgt_score,
        test_dS=test_dS,
        test_dT=test_dT,
        test_score=test_score,
        all_scores=all_scores,
        all_is_train=all_is_train,
        all_is_test=all_is_test,
        all_origin=all_origin,
        all_file_ids=all_ids,
    )


# =============================================================================
# 5. 1200-POINT NATURAL FAR-GROUP OBJECTIVE
# =============================================================================

def evaluate_best_cut(bundle: ScoreBundle) -> Dict[str, float]:
    scores = np.asarray(bundle.all_scores, dtype=float)
    n_total = len(scores)
    n_train = int(bundle.all_is_train.sum())
    n_test = int(bundle.all_is_test.sum())

    if n_total < 10:
        raise ValueError("too few scores")

    order = np.argsort(scores, kind="mergesort")
    s = scores[order]
    is_train_sorted = bundle.all_is_train[order]
    is_test_sorted = bundle.all_is_test[order]

    hmin = max(1, min(HIGH_GROUP_MIN, n_total - 1))
    hmax = max(hmin, min(HIGH_GROUP_MAX, n_total - 1))

    best = None

    for high_size in range(hmin, hmax + 1):
        cut = n_total - high_size
        low = s[:cut]
        high = s[cut:]

        low_scale = robust_scale_1d(low)
        low_med = float(np.median(low))
        high_med = float(np.median(high))

        separation = (high_med - low_med) / low_scale
        boundary_gap = float(s[cut] - s[cut - 1]) / low_scale

        low_q95 = float(np.quantile(low, 0.95))
        high_q25 = float(np.quantile(high, 0.25))
        tail_separation = (high_q25 - low_q95) / low_scale

        high_train = int(np.sum(is_train_sorted[cut:]))
        high_test = int(np.sum(is_test_sorted[cut:]))

        high_test_purity = high_test / max(high_size, 1)
        train_contamination_rate = high_train / max(n_train, 1)
        test_capture_rate = high_test / max(n_test, 1)

        # Soft group-size preference. h=50 or 150 is still allowed and gets
        # meaningful credit; this is not a hard 100-count assumption.
        size_sigma = max(float(EXPECTED_HIGH_GROUP) * 0.50, 1.0)
        size_preference = math.exp(
            -0.5 * ((high_size - EXPECTED_HIGH_GROUP) / size_sigma) ** 2
        )

        geometry_score = (
            W_SEPARATION * safe_log_positive(separation)
            + W_BOUNDARY_GAP * safe_log_positive(boundary_gap)
            + W_TAIL_SEPARATION * safe_log_positive(tail_separation)
        )

        # A geometrically beautiful tail made mostly of KNOWN NORMAL TRAIN
        # points is NOT an anomaly candidate. Gate geometry by test purity and
        # strongly penalize the fraction of the high group that is known-normal.
        high_train_fraction = high_train / max(high_size, 1)
        geometry_purity_factor = 0.20 + 0.80 * high_test_purity
        geometry_effective = geometry_score * geometry_purity_factor

        composition_score = (
            W_HIGH_TEST_PURITY * high_test_purity
            + W_TEST_CAPTURE * test_capture_rate
            - W_KNOWN_NORMAL_CONTAM * train_contamination_rate
            - W_HIGH_TRAIN_FRACTION * high_train_fraction
            + W_SIZE_PREFERENCE * size_preference
        )

        fitness = geometry_effective + composition_score

        rec = {
            "fitness": float(fitness),
            "high_size": int(high_size),
            "low_size": int(cut),
            "cut_threshold": float(0.5 * (s[cut - 1] + s[cut])),
            "boundary_low_score": float(s[cut - 1]),
            "boundary_high_score": float(s[cut]),
            "separation": float(separation),
            "boundary_gap": float(boundary_gap),
            "tail_separation": float(tail_separation),
            "high_train_count": int(high_train),
            "high_test_count": int(high_test),
            "high_test_purity": float(high_test_purity),
            "train_contamination_rate": float(train_contamination_rate),
            "test_capture_rate": float(test_capture_rate),
            "size_preference": float(size_preference),
            "geometry_score": float(geometry_score),
            "geometry_purity_factor": float(geometry_purity_factor),
            "geometry_effective": float(geometry_effective),
            "high_train_fraction": float(high_train_fraction),
            "composition_score": float(composition_score),
            "low_median": float(low_med),
            "high_median": float(high_med),
            "low_scale": float(low_scale),
            "low_q95": float(low_q95),
            "high_q25": float(high_q25),
        }

        # Main objective, then purity, then less train leakage, then geometry.
        key = (
            rec["fitness"],
            rec["high_test_purity"],
            -rec["train_contamination_rate"],
            rec["separation"],
            rec["boundary_gap"],
            -abs(rec["high_size"] - EXPECTED_HIGH_GROUP),
        )

        if best is None or key > best[0]:
            best = (key, rec)

    if best is None:
        raise RuntimeError("No valid high-group cut")
    return best[1]


# =============================================================================
# 6. SUBSET ENGINE WITH SCALAR CACHE
# =============================================================================

class SubsetEngine:
    def __init__(self, data: PreparedMachine):
        self.data = data
        self.cache: Dict[Tuple[int, ...], Dict[str, object]] = {}

    def evaluate(self, subset: Tuple[int, ...]) -> Dict[str, object]:
        subset = tuple(sorted(int(j) for j in subset))
        if subset in self.cache:
            return self.cache[subset]

        bundle = compute_score_bundle(self.data, subset)
        cut = evaluate_best_cut(bundle)

        result: Dict[str, object] = {
            "subset": subset,
            "dim": len(subset),
            **cut,
        }
        self.cache[subset] = result
        return result


def within_dim_ranking_key(r: Dict[str, object]) -> Tuple:
    return (
        float(r["fitness"]),
        float(r["high_test_purity"]),
        -float(r["train_contamination_rate"]),
        float(r["separation"]),
        float(r["boundary_gap"]),
        -abs(int(r["high_size"]) - EXPECTED_HIGH_GROUP),
    )


def final_cross_dim_key(r: Dict[str, object]) -> Tuple:
    return (
        float(r.get("dim_z", -999.0)),
        float(r["fitness"]),
        float(r["high_test_purity"]),
        -float(r["train_contamination_rate"]),
        float(r["separation"]),
        -int(r["dim"]),
    )


def result_to_row(machine: str, r: Dict[str, object], rank: Optional[int] = None) -> Dict[str, object]:
    subset = tuple(r["subset"])
    row = {
        "machine": machine,
        "rank": rank,
        "dim": int(r["dim"]),
        "dim_z": float(r.get("dim_z", np.nan)),
        "label_free_fitness": float(r["fitness"]),
        "high_size": int(r["high_size"]),
        "low_size": int(r["low_size"]),
        "cut_threshold": float(r["cut_threshold"]),
        "separation": float(r["separation"]),
        "boundary_gap": float(r["boundary_gap"]),
        "tail_separation": float(r["tail_separation"]),
        "high_train_count": int(r["high_train_count"]),
        "high_test_count": int(r["high_test_count"]),
        "high_test_purity": float(r["high_test_purity"]),
        "train_contamination_rate": float(r["train_contamination_rate"]),
        "test_capture_rate": float(r["test_capture_rate"]),
        "features": ";".join(BEST_FEATURE_POOL_68[j] for j in subset),
        "feature_indices": ";".join(str(j) for j in subset),
    }
    return row


# =============================================================================
# 7. DIMENSION-CALIBRATED DEEP SEARCH D=1..20
# =============================================================================

def random_subset(dim: int, rng: np.random.Generator) -> Tuple[int, ...]:
    idx = rng.choice(len(BEST_FEATURE_POOL_68), size=dim, replace=False)
    return tuple(sorted(int(v) for v in idx))


def build_expansion_pool(
    singles: Sequence[Dict[str, object]],
    pairs: Sequence[Dict[str, object]],
) -> List[int]:
    votes = {j: 0.0 for j in range(len(BEST_FEATURE_POOL_68))}

    for rank, r in enumerate(singles[:TOP_SINGLES_FOR_POOL]):
        weight = TOP_SINGLES_FOR_POOL - rank
        for j in r["subset"]:
            votes[int(j)] += 5.0 * weight

    for rank, r in enumerate(pairs[:TOP_PAIRS_FOR_POOL]):
        weight = TOP_PAIRS_FOR_POOL - rank
        for j in r["subset"]:
            votes[int(j)] += weight

    ranked = sorted(votes, key=lambda j: (-votes[j], j))
    return ranked[:min(EXPANSION_POOL_SIZE, len(ranked))]


def dimension_baseline(
    engine: SubsetEngine,
    dim: int,
    known_results: Sequence[Dict[str, object]],
    rng: np.random.Generator,
) -> List[float]:
    if dim in (1, 2):
        return [float(r["fitness"]) for r in known_results]

    vals = []
    seen = set()
    attempts = 0
    max_attempts = CALIBRATION_RANDOM_PER_DIM * 20
    while len(vals) < CALIBRATION_RANDOM_PER_DIM and attempts < max_attempts:
        attempts += 1
        s = random_subset(dim, rng)
        if s in seen:
            continue
        seen.add(s)
        vals.append(float(engine.evaluate(s)["fitness"]))

    if len(vals) < 10:
        # Fallback to the dimension's explored results.
        vals.extend(float(r["fitness"]) for r in known_results)
    return vals


def attach_dimension_z(
    results: Sequence[Dict[str, object]],
    baseline: Sequence[float],
) -> None:
    for r in results:
        r["dim_z"] = robust_z(float(r["fitness"]), baseline)


def search_one_machine(data: PreparedMachine) -> Dict[str, object]:
    t0 = time.time()
    machine = data.machine
    out_dir = os.path.join(PER_MACHINE_DIR, machine)
    ensure_dir(out_dir)

    print("\n" + "#" * 120)
    print(f"SEARCHING {machine}: 1200-POINT DUAL RAW-kNN CLUSTER SEARCH")
    print("#" * 120)

    engine = SubsetEngine(data)
    rng = np.random.default_rng(RANDOM_SEED + sum(ord(c) for c in machine))

    dim_results: Dict[int, List[Dict[str, object]]] = {}
    best_by_dim: List[Dict[str, object]] = []

    # -------------------------------------------------------------------------
    # D=1 exhaustive
    # -------------------------------------------------------------------------
    print("\n[D=1] exhaustive 68 singles")
    singles = [engine.evaluate((j,)) for j in range(68)]
    singles.sort(key=within_dim_ranking_key, reverse=True)
    dim_results[1] = singles
    base1 = dimension_baseline(engine, 1, singles, rng)
    attach_dimension_z(singles, base1)
    best_by_dim.append(singles[0])
    print(
        f"best D=1 raw={singles[0]['fitness']:.4f} "
        f"z={singles[0]['dim_z']:.3f} "
        f"high={singles[0]['high_size']} "
        f"purity={singles[0]['high_test_purity']:.3f}"
    )

    # -------------------------------------------------------------------------
    # D=2 exhaustive
    # -------------------------------------------------------------------------
    print("\n[D=2] exhaustive 2278 pairs")
    pairs: List[Dict[str, object]] = []
    count = 0
    for a in range(68):
        for b in range(a + 1, 68):
            pairs.append(engine.evaluate((a, b)))
            count += 1
            if count % 400 == 0:
                print(f"  pairs {count}/2278")

    pairs.sort(key=within_dim_ranking_key, reverse=True)
    dim_results[2] = pairs
    base2 = dimension_baseline(engine, 2, pairs, rng)
    attach_dimension_z(pairs, base2)
    best_by_dim.append(pairs[0])
    print(
        f"best D=2 raw={pairs[0]['fitness']:.4f} "
        f"z={pairs[0]['dim_z']:.3f} "
        f"high={pairs[0]['high_size']} "
        f"purity={pairs[0]['high_test_purity']:.3f}"
    )

    expansion_pool = build_expansion_pool(singles, pairs)
    print("\nExpansion pool:")
    for j in expansion_pool:
        print(f"  {j:02d} {BEST_FEATURE_POOL_68[j]}")

    # Beam starts from top D=2 candidates.
    beam = pairs[:BEAM_WIDTH]

    # -------------------------------------------------------------------------
    # D=3..MAX_DIM beam + random injection
    # -------------------------------------------------------------------------
    for dim in range(3, MAX_DIM + 1):
        print(f"\n[D={dim}] beam search")

        candidate_subsets: List[Tuple[int, ...]] = []
        for parent in beam:
            parent_set = set(int(v) for v in parent["subset"])
            for j in expansion_pool:
                if j not in parent_set:
                    candidate_subsets.append(tuple(sorted((*parent["subset"], j))))

        # Random injections reduce the chance that forward beam gets trapped.
        for _ in range(RANDOM_INJECTIONS_PER_DIM):
            candidate_subsets.append(random_subset(dim, rng))

        candidate_subsets = unique_tuples(candidate_subsets)
        print(f"  evaluating {len(candidate_subsets)} unique candidates")

        current = []
        for i, subset in enumerate(candidate_subsets, start=1):
            current.append(engine.evaluate(subset))
            if i % 250 == 0:
                print(f"    {i}/{len(candidate_subsets)}")

        current.sort(key=within_dim_ranking_key, reverse=True)
        # Keep a generous diagnostic slice; beam itself remains BEAM_WIDTH.
        dim_results[dim] = current[:max(TOP_KEEP_PER_DIM, BEAM_WIDTH)]

        baseline = dimension_baseline(engine, dim, current, rng)
        attach_dimension_z(dim_results[dim], baseline)
        # Ensure best candidate z is attached even if object references differ.
        attach_dimension_z(current[:1], baseline)
        best_by_dim.append(current[0])

        print(
            f"best D={dim} raw={current[0]['fitness']:.4f} "
            f"z={current[0]['dim_z']:.3f} "
            f"high={current[0]['high_size']} "
            f"trainHigh={current[0]['high_train_count']} "
            f"testHigh={current[0]['high_test_count']} "
            f"purity={current[0]['high_test_purity']:.3f}"
        )

        beam = current[:BEAM_WIDTH]

    # -------------------------------------------------------------------------
    # Save best by dimension.
    # -------------------------------------------------------------------------
    best_dim_rows = []
    for dim in range(1, MAX_DIM + 1):
        r = max(dim_results[dim], key=within_dim_ranking_key)
        best_dim_rows.append(result_to_row(machine, r))

    best_dim_df = pd.DataFrame(best_dim_rows)
    best_dim_df.to_csv(
        os.path.join(out_dir, f"{machine}_best_by_dimension.csv"),
        index=False,
    )

    # -------------------------------------------------------------------------
    # Cross-dimension candidate pool and TOP 10.
    # -------------------------------------------------------------------------
    global_candidates: List[Dict[str, object]] = []
    for dim in range(1, MAX_DIM + 1):
        # D1/D2 can be large; top 80 is more than enough for global top-10.
        pool = sorted(dim_results[dim], key=final_cross_dim_key, reverse=True)
        global_candidates.extend(pool[:TOP_KEEP_PER_DIM])

    # Deduplicate exact subsets.
    dedup: Dict[Tuple[int, ...], Dict[str, object]] = {}
    for r in global_candidates:
        s = tuple(r["subset"])
        if s not in dedup or final_cross_dim_key(r) > final_cross_dim_key(dedup[s]):
            dedup[s] = r

    ranked = sorted(dedup.values(), key=final_cross_dim_key, reverse=True)
    top10 = ranked[:TOP_N_SUBSETS]

    if len(top10) < TOP_N_SUBSETS:
        raise RuntimeError(f"{machine}: only {len(top10)} final candidates")

    top_rows = [result_to_row(machine, r, rank=i + 1) for i, r in enumerate(top10)]
    pd.DataFrame(top_rows).to_csv(
        os.path.join(out_dir, f"{machine}_top10_subsets.csv"),
        index=False,
    )

    # -------------------------------------------------------------------------
    # Feature effectiveness ranking across top candidates + best dimensions.
    # Each candidate contributes 1/dim total mass so larger subsets do not win
    # merely because they contain more features.
    # -------------------------------------------------------------------------
    votes = {j: 0.0 for j in range(68)}
    appearances = {j: 0 for j in range(68)}
    best_dim_appear = {j: 0 for j in range(68)}

    for rank, r in enumerate(top10, start=1):
        rank_weight = (TOP_N_SUBSETS - rank + 1) / TOP_N_SUBSETS
        per_feature = rank_weight / max(int(r["dim"]), 1)
        for j in r["subset"]:
            votes[int(j)] += per_feature
            appearances[int(j)] += 1

    for r in best_by_dim:
        per_feature = 1.0 / max(int(r["dim"]), 1)
        for j in r["subset"]:
            votes[int(j)] += 0.25 * per_feature
            best_dim_appear[int(j)] += 1

    feat_rows = []
    for j in range(68):
        feat_rows.append({
            "machine": machine,
            "feature": BEST_FEATURE_POOL_68[j],
            "effectiveness_vote": float(votes[j]),
            "top10_appearances": int(appearances[j]),
            "best_dimension_appearances": int(best_dim_appear[j]),
        })

    feat_df = pd.DataFrame(feat_rows).sort_values(
        ["effectiveness_vote", "top10_appearances"],
        ascending=[False, False],
    )
    feat_df.insert(0, "feature_rank", np.arange(1, len(feat_df) + 1))
    feat_df.to_csv(
        os.path.join(out_dir, f"{machine}_feature_effectiveness_ranking.csv"),
        index=False,
    )


    elapsed = time.time() - t0
    print(f"\n{machine} search complete. evaluated subsets={len(engine.cache)}")
    print(f"elapsed_sec={elapsed:.1f}")
    print("TOP 10:")
    print(pd.DataFrame(top_rows)[[
        "rank", "dim", "dim_z", "label_free_fitness", "high_size",
        "high_train_count", "high_test_count", "high_test_purity", "features"
    ]].to_string(index=False))

    return {
        "machine": machine,
        "top10": top10,
        "best_by_dim": best_by_dim,
        "feature_ranking": feat_df,
        "evaluated_subset_count": len(engine.cache),
        "elapsed_sec": elapsed,
    }


# =============================================================================
# 8. GLOBAL OUTPUT SUMMARIES
# =============================================================================


# =============================================================================

def save_global_search_summaries(search_results: Dict[str, Dict[str, object]]) -> None:
    top_rows = []
    dim_rows = []
    feat_frames = []

    for machine in MACHINE_TYPES:
        res = search_results[machine]
        for rank, r in enumerate(res["top10"], start=1):
            top_rows.append(result_to_row(machine, r, rank=rank))

        for r in res["best_by_dim"]:
            dim_rows.append(result_to_row(machine, r))

        feat_frames.append(res["feature_ranking"])

    pd.DataFrame(top_rows).to_csv(
        os.path.join(OUTPUT_DIR, "selected_top10_by_machine.csv"),
        index=False,
    )
    pd.DataFrame(dim_rows).to_csv(
        os.path.join(OUTPUT_DIR, "best_by_dimension.csv"),
        index=False,
    )
    pd.concat(feat_frames, ignore_index=True).to_csv(
        os.path.join(OUTPUT_DIR, "feature_effectiveness_ranking.csv"),
        index=False,
    )

    # Machine-independent frequency summary over 8x10 selected subsets.
    global_votes = {f: 0.0 for f in BEST_FEATURE_POOL_68}
    global_apps = {f: 0 for f in BEST_FEATURE_POOL_68}

    for machine in MACHINE_TYPES:
        for rank, r in enumerate(search_results[machine]["top10"], start=1):
            rank_weight = (TOP_N_SUBSETS - rank + 1) / TOP_N_SUBSETS
            per_feature = rank_weight / max(int(r["dim"]), 1)
            for j in r["subset"]:
                f = BEST_FEATURE_POOL_68[int(j)]
                global_votes[f] += per_feature
                global_apps[f] += 1

    rows = [
        {
            "feature": f,
            "global_effectiveness_vote": global_votes[f],
            "appearances_across_8x10": global_apps[f],
        }
        for f in BEST_FEATURE_POOL_68
    ]
    global_feat = pd.DataFrame(rows).sort_values(
        ["global_effectiveness_vote", "appearances_across_8x10"],
        ascending=[False, False],
    ).reset_index(drop=True)
    global_feat.insert(0, "global_rank", np.arange(1, len(global_feat) + 1))
    global_feat.to_csv(
        os.path.join(OUTPUT_DIR, "global_feature_effectiveness_ranking.csv"),
        index=False,
    )


def save_method_config() -> None:
    config = {
        "method": "Frozen68 pooled-normal MinMax + raw Dual kNN min + 1200-point natural far-cluster transductive search",
        "selection_is_transductive_unlabeled_test": True,
        "test_anomaly_labels_used_during_selection": False,
        "test_domain_labels_used_during_selection": False,
        "known_normal_train_used": True,
        "candidate_pool_size": 68,
        "max_dimension": MAX_DIM,
        "source_k": SOURCE_K,
        "target_k": TARGET_K,
        "distance_domain_normalization_q95": False,
        "final_anomaly_score": "min(raw_source_knn_distance, raw_target_knn_distance)",
        "feature_scaler": SCALER_MODE,
        "feature_scaler_fit_data": "pooled 990 source-normal + 10 target-normal train",
        "test_scaling_clipped": False,
        "distance_divide_by_sqrt_dimension": DISTANCE_DIM_NORMALIZE,
        "high_group_min": HIGH_GROUP_MIN,
        "high_group_max": HIGH_GROUP_MAX,
        "expected_high_group_soft_center": EXPECTED_HIGH_GROUP,
        "search": {
            "D1": "exhaustive 68 singles",
            "D2": "exhaustive 2278 pairs",
            "D3_to_D20": "beam forward + random injections",
            "beam_width": BEAM_WIDTH,
            "expansion_pool_size": EXPANSION_POOL_SIZE,
            "random_injections_per_dim": RANDOM_INJECTIONS_PER_DIM,
            "calibration_random_per_dim": CALIBRATION_RANDOM_PER_DIM,
        },
        "dimension_comparison": "robust z relative to same-dimensional baseline",
        "objective_weights": {
            "separation": W_SEPARATION,
            "boundary_gap": W_BOUNDARY_GAP,
            "tail_separation": W_TAIL_SEPARATION,
            "high_test_purity": W_HIGH_TEST_PURITY,
            "test_capture": W_TEST_CAPTURE,
            "known_normal_contamination_rate_penalty": W_KNOWN_NORMAL_CONTAM,
            "known_normal_high_group_fraction_penalty": W_HIGH_TRAIN_FRACTION,
            "size_preference": W_SIZE_PREFERENCE,
        },
        "top_n_subsets_reported_per_machine": TOP_N_SUBSETS,
        "fixed68": BEST_FEATURE_POOL_68,
    }

    with open(
        os.path.join(OUTPUT_DIR, "search_method_config.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(config, f, indent=2)


def create_output_zip() -> str:
    base = OUTPUT_DIR.rstrip("/\\")
    zip_path = shutil.make_archive(base, "zip", OUTPUT_DIR)
    print("\nZIP CREATED:", zip_path)
    return zip_path


def maybe_auto_download(path: str) -> None:
    if not AUTO_DOWNLOAD_ZIP:
        return
    try:
        from google.colab import files as colab_files
        colab_files.download(path)
    except Exception as exc:
        print("Automatic Colab download skipped:", exc)
        print("Output ZIP:", path)


# =============================================================================
# 9. MAIN -- FEATURE SELECTION / REPORTING ONLY
# =============================================================================

def save_selected_feature_reports(search_results: Dict[str, Dict[str, object]]) -> None:
    """Save compact machine-specific selected-feature reports."""
    top1_rows = []
    long_rows = []
    selected_json: Dict[str, List[Dict[str, object]]] = {}
    top1_json: Dict[str, Dict[str, object]] = {}

    for machine in MACHINE_TYPES:
        selected_json[machine] = []

        for rank, r in enumerate(search_results[machine]["top10"], start=1):
            features = [BEST_FEATURE_POOL_68[int(j)] for j in r["subset"]]
            selected_json[machine].append({
                "rank": rank,
                "dim": int(r["dim"]),
                "dim_z": float(r["dim_z"]),
                "label_free_fitness": float(r["fitness"]),
                "high_group_size": int(r["high_size"]),
                "high_train_count": int(r["high_train_count"]),
                "high_test_count": int(r["high_test_count"]),
                "high_test_purity": float(r["high_test_purity"]),
                "features": features,
            })

        best = search_results[machine]["top10"][0]
        best_features = [BEST_FEATURE_POOL_68[int(j)] for j in best["subset"]]

        top1_rows.append({
            "machine": machine,
            "dim": int(best["dim"]),
            "dim_z": float(best["dim_z"]),
            "label_free_fitness": float(best["fitness"]),
            "high_group_size": int(best["high_size"]),
            "high_train_count": int(best["high_train_count"]),
            "high_test_count": int(best["high_test_count"]),
            "high_test_purity": float(best["high_test_purity"]),
            "features": ";".join(best_features),
        })

        top1_json[machine] = {
            "dim": int(best["dim"]),
            "dim_z": float(best["dim_z"]),
            "label_free_fitness": float(best["fitness"]),
            "high_group_size": int(best["high_size"]),
            "high_train_count": int(best["high_train_count"]),
            "high_test_count": int(best["high_test_count"]),
            "high_test_purity": float(best["high_test_purity"]),
            "features": best_features,
        }

        for feature_order, feature in enumerate(best_features, start=1):
            long_rows.append({
                "machine": machine,
                "feature_order": feature_order,
                "feature": feature,
            })

    pd.DataFrame(top1_rows).to_csv(
        os.path.join(OUTPUT_DIR, "selected_top1_by_machine.csv"),
        index=False,
    )
    pd.DataFrame(long_rows).to_csv(
        os.path.join(OUTPUT_DIR, "selected_top1_features_long.csv"),
        index=False,
    )

    with open(
        os.path.join(OUTPUT_DIR, "selected_top10_by_machine.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(selected_json, f, indent=2)

    with open(
        os.path.join(OUTPUT_DIR, "selected_top1_by_machine.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(top1_json, f, indent=2)


def main() -> None:
    set_global_seed()
    clean_output_dir()
    save_method_config()

    print("=" * 120)
    print("DCASE 2025 TASK 2 -- FROZEN-68 / 1200-POINT FEATURE SELECTION ONLY")
    print("=" * 120)
    print(f"INPUT_DIR                  : {INPUT_DIR}")
    print(f"OUTPUT_DIR                 : {OUTPUT_DIR}")
    print(f"Frozen feature pool        : {len(BEST_FEATURE_POOL_68)}")
    print(f"Source k / Target k        : {SOURCE_K} / {TARGET_K}")
    print("Domain distance q95 norm   : NO")
    print("Final anomaly score        : min(d_source, d_target)")
    print(f"Feature scaling            : {SCALER_MODE}, pooled normal train, NO CLIP")
    print(f"sqrt(D) distance normalize : {DISTANCE_DIM_NORMALIZE}")
    print(f"Search dimensions          : 1..{MAX_DIM}")
    print(f"Far-group size scan        : {HIGH_GROUP_MIN}..{HIGH_GROUP_MAX}")
    print(f"Top subsets reported       : {TOP_N_SUBSETS}")
    print("TEST LABELS IN SEARCH      : NO")
    print("TEST DOMAIN IN SEARCH      : NO")
    print("UNLABELED TEST FEATURES    : YES (TRANSDUCTIVE)")
    print("OFFICIAL EVALUATOR         : DISABLED / REMOVED")

    # Prepare all machines first; fail immediately on missing files/features.
    prepared: Dict[str, PreparedMachine] = {}
    for machine in MACHINE_TYPES:
        prepared[machine] = prepare_machine(machine)

    # Label-free/transductive feature-subset search.
    search_results: Dict[str, Dict[str, object]] = {}
    total_t0 = time.time()

    for machine in MACHINE_TYPES:
        search_results[machine] = search_one_machine(prepared[machine])

    # Feature reports only.
    save_global_search_summaries(search_results)
    save_selected_feature_reports(search_results)

    total_elapsed = time.time() - total_t0

    print("\n" + "#" * 120)
    print("FINAL LABEL-FREE SELECTED TOP-1 FEATURE SUBSET PER MACHINE")
    print("#" * 120)

    top1_rows = []
    for machine in MACHINE_TYPES:
        r = search_results[machine]["top10"][0]
        top1_rows.append(result_to_row(machine, r, rank=1))

    top1_df = pd.DataFrame(top1_rows)
    print(top1_df[[
        "machine", "dim", "dim_z", "label_free_fitness", "high_size",
        "high_train_count", "high_test_count", "high_test_purity", "features"
    ]].to_string(index=False))

    print(f"\nTotal feature-search elapsed_sec={total_elapsed:.1f}")
    print("Feature-report directory:", OUTPUT_DIR)
    print("No official evaluator or official ranking was run.")

    zip_path = create_output_zip()
    maybe_auto_download(zip_path)


if __name__ == "__main__":
    main()